# 03 - Algorithmes GDS (Graph Data Science)

Ce notebook explore les algorithmes de la bibliotheque Neo4j GDS.

## Algorithmes couverts
- Louvain (detection de communautes)
- PageRank (centralite)
- Betweenness Centrality
- Shortest Path

In [2]:
import os
from neo4j import GraphDatabase
import pandas as pd

NEO4J_URI = os.environ.get('NEO4J_URI', 'bolt://neo4j:7687')
driver = GraphDatabase.driver(
    NEO4J_URI, 
    auth=(os.environ.get('NEO4J_USER', 'neo4j'), 
          os.environ['NEO4J_PASSWORD'])
)

def query(cypher, params=None):
    with driver.session() as session:
        result = session.run(cypher, params or {})
        return pd.DataFrame([dict(r) for r in result])

## 1. Creation de la projection GDS

Avant d'utiliser les algorithmes, il faut projeter le graphe en memoire.

In [ ]:
# Supprimer projection existante si besoin
try:
    query("CALL gds.graph.drop('epidemic-network', false)")
except:
    pass

# Creer la projection
query("""
CALL gds.graph.project(
  'epidemic-network',
  'Patient',
  'INFECTED_BY'
) YIELD graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount
""")

## 2. Louvain - Detection de communautes

L'algorithme Louvain detecte les clusters de noeuds fortement connectes.

In [3]:
df_louvain = query("""
CALL gds.louvain.stream('epidemic-network')
YIELD nodeId, communityId
WITH communityId, count(*) AS size
RETURN communityId, size
ORDER BY size DESC
LIMIT 15
""")
print(f"Nombre de clusters: {len(df_louvain)}")
df_louvain

Nombre de clusters: 15


,communityId,size
0,35045,5000
1,22509,5000
2,17509,5000
3,7288,4945
4,1760,2018
5,6488,800
6,0,24
7,3778,20
8,6,1
9,11,1


## 3. PageRank - Identification des super-propagateurs

PageRank mesure l'importance d'un noeud dans le reseau.

In [ ]:
query("""
CALL gds.pageRank.stream('epidemic-network')
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS patient, score
RETURN patient.patient_id AS id, patient.age AS age, score
ORDER BY score DESC
LIMIT 15
""")

## 4. Betweenness Centrality - Points de blocage

Identifie les noeuds qui servent de "ponts" entre communautes.

In [ ]:
query("""
CALL gds.betweenness.stream('epidemic-network')
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS patient, score
WHERE score > 0
RETURN patient.patient_id AS id, score
ORDER BY score DESC
LIMIT 10
""")

## 5. Shortest Path - Chemin le plus court

In [ ]:
# Trouver le chemin entre deux patients
query("""
MATCH (source:Patient {patient_id: 'P000000'}),
      (target:Patient {patient_id: 'P000050'}),
      path = shortestPath((source)-[:INFECTED_BY*]-(target))
RETURN [n IN nodes(path) | n.patient_id] AS chemin, length(path) AS hops
""")

In [ ]:
driver.close()